## Section A

#### A1. (5 pts) 
Why must the tokenizer and the model weights come from the same repo_id? Describe what would go wrong if you paired Phi-3's weights with a different model's tokenizer.

#### Answer: 
The models comes with its own tokenizer, if we use different tokenizer, it will fail because the tokenization algorithm and number might be different from how the model was originally trained.


==========================================================================================


#### A2. (5 pts) 
The embedding layer is nn.Embedding(vocab_size, d_model). Explain the role of each of the two numbers. Why is the first one vocab_size and not something else?

#### Answer: 
vocab_size is the vocabulary size of the tokenizer that comes with the model and the d_model is the dimensions in which we want to project our vectors. We use the vocabulary size and the dimension to create embeddings. It must be vocab_size first because there must be one embedding row for each possible token ID.


==========================================================================================


#### A3. (5 pts) 
In self-attention we create three projections (Query, Key, Value) from the same input embedding. In one sentence each, state the "job" of Q, K, and V.

#### Answer: 
Query: What am I looking for? What this vector wants to find in the other words.
Key: What do I offer? a labal advertising what this word is about, so other words Queries can match against it.
Value: What content do I carry - what actual information that gets passed away along when a word is attended to.


==========================================================================================


#### A4. (5 pts) 
Why is a dot product a sensible way to measure how much one word should attend to another? What does a large dot product imply about the two vectors?

#### Answer: 
A large dot product means query and key are strongly alligned and highly relevance and that word recieves more attention.


==========================================================================================


#### A5. (5 pts) 
After softmax, each row of the attention matrix sums to 1.0. Why is this property desirable? What would it mean if a row summed to 3.0 instead?

#### Answer: 
Rows summing to 1.0 means the attention weights form a probability distribution — a set of percentages. This lets you interpret them as "spend 40% of attention here, 20% there…" and, crucially, it makes the output a proper weighted average of the Values (weights that sum to 1 = a genuine average, keeping the output on the same scale as the inputs).

In [9]:
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer
import torch.nn as nn

#### C1. (10 pts) Write a function:

In [16]:
def scaled_dot_product_attention(query, key, value, d_model):
    dot_products = torch.matmul(query, key.transpose(-2,-1))
    scores = F.softmax(dot_products / np.sqrt(d_model), dim=-1)
    context = torch.matmul(scores, value)
    return context, scores

In [ ]:
# repo_id
# tokenizer
# reproduciblity
# embedding dim
# embedding layer
# sentence
# tokenize sentence into tensor
# create embedding of the tokenized sentence
# linear project of q,k,v
# projection for q,k,v
# pass q,k,v projection to the function scaled_dot_product_attention


In [21]:
repo_id = 'microsoft/Phi-3-mini-4k-instruct'
tokenizer = AutoTokenizer.from_pretrained(repo_id)
vocab_size = len(tokenizer)
torch.manual_seed(13)

d_model = 1024

embedding_layer = nn.Embedding(vocab_size, d_model)
sentence = 'Just a dummy sentence' 
input_ids = tokenizer(sentence, return_tensors='pt')['input_ids']
embeddings = embedding_layer(input_ids)
linear_q = nn.Linear(d_model, d_model)
linear_k = nn.Linear(d_model, d_model)
linear_v = nn.Linear(d_model, d_model)

proj_k = linear_k(embeddings)
proj_q = linear_q(embeddings)
proj_v = linear_v(embeddings)

context, scores = scaled_dot_product_attention(proj_q, proj_k, proj_v, d_model)

print("context shape:   ", context.shape)

# (2) confirm it matches the input embeddings
print("embeddings shape:", embeddings.shape)
print("match:", context.shape == embeddings.shape)



context shape:    torch.Size([1, 4, 1024])
embeddings shape: torch.Size([1, 4, 1024])
match: True
